# 08 - Error Analysis and Guardrails

This notebook audits the latest 200-example baseline metrics and the uploaded 1,200-example LoRA generation test.

In [ ]:
from pathlib import Path

import pandas as pd

from src.agents import GuardrailAgent
from src.evaluation import compute_bleu, compute_rouge

OUTPUT_DIR = Path('../outputs') if Path.cwd().name == 'notebooks' else Path('outputs')
comparison_df = pd.read_csv(OUTPUT_DIR / 'baseline_200_comparison_table.csv')
task_df = pd.read_csv(OUTPUT_DIR / 'baseline_200_task_metrics.csv')
lora_metrics_df = pd.read_csv(OUTPUT_DIR / 'lora_test_metrics.csv')
lora_generations_df = pd.read_csv(OUTPUT_DIR / 'lora_test_generations.csv')

display(comparison_df)
display(lora_metrics_df)

## Apply Output Guardrails to LoRA Test Generations

The uploaded LoRA generation file has `reference` and `prediction` only, so this notebook can detect repetition, short answers, and unsupported-claim proxies, but cannot fully verify grounding against retrieved evidence.

In [ ]:
guardrail = GuardrailAgent()
rows = []
for idx, row in enumerate(lora_generations_df.itertuples(index=False), start=1):
    reference = '' if pd.isna(row.reference) else str(row.reference)
    prediction = '' if pd.isna(row.prediction) else str(row.prediction)
    rouge = compute_rouge(reference, prediction)
    bleu = compute_bleu(reference, prediction)
    result = guardrail.validate_output(prediction)
    messages = [finding.message for finding in result.findings]
    failure_types = []
    if len(prediction.split()) < 12:
        failure_types.append('too_short')
    if any('Repetitive wording' in message for message in messages):
        failure_types.append('repetition')
    if rouge['rougeL'] < 0.20:
        failure_types.append('low_rougeL')
    if bleu < 0.05:
        failure_types.append('low_bleu')
    if any('strong quantitative' in message or 'Citation-like' in message for message in messages):
        failure_types.append('hallucination_risk')
    if not failure_types:
        failure_types.append('no_major_proxy_failure')
    rows.append({
        'example_id': idx,
        'strategy': 'fine_tuned_lora_test',
        'reference': reference,
        'candidate': prediction,
        'bleu': bleu,
        'rouge1': rouge['rouge1'],
        'rouge2': rouge['rouge2'],
        'rougeL': rouge['rougeL'],
        'word_count': len(prediction.split()),
        'guardrail_findings': ' | '.join(messages),
        'failure_types': ', '.join(failure_types),
    })
analysis_df = pd.DataFrame(rows)
analysis_df.head()

In [ ]:
failure_summary = (
    analysis_df.assign(failure_type=analysis_df['failure_types'].str.split(', '))
    .explode('failure_type')
    .groupby(['strategy', 'failure_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['strategy', 'count'], ascending=[True, False])
)

finding_summary = (
    analysis_df[analysis_df['guardrail_findings'].ne('')]
    .assign(message=analysis_df['guardrail_findings'].str.split(' | '))
    .explode('message')
    .groupby(['strategy', 'message'])
    .size()
    .reset_index(name='count')
    .sort_values(['strategy', 'count'], ascending=[True, False])
)

display(failure_summary)
display(finding_summary)

In [ ]:
worst_cases = analysis_df.sort_values(['rougeL', 'bleu'], ascending=[True, True]).head(10)
best_cases = analysis_df.sort_values(['rougeL', 'bleu'], ascending=[False, False]).head(10)

display(worst_cases[['example_id', 'bleu', 'rougeL', 'failure_types', 'candidate', 'reference']])
display(best_cases[['example_id', 'bleu', 'rougeL', 'candidate', 'reference']])

In [ ]:
analysis_df.to_csv(OUTPUT_DIR / 'error_analysis_cases.csv', index=False)
failure_summary.to_csv(OUTPUT_DIR / 'error_analysis_failure_summary.csv', index=False)
finding_summary.to_csv(OUTPUT_DIR / 'error_analysis_guardrail_summary.csv', index=False)
worst_cases.to_csv(OUTPUT_DIR / 'lora_test_worst_cases.csv', index=False)
best_cases.to_csv(OUTPUT_DIR / 'lora_test_best_cases.csv', index=False)
print('Updated error-analysis artifacts in', OUTPUT_DIR)